# Sequential Measurement Strategy Benchmark

Compares three strategies for selecting liquidus measurement compositions, using a system with a known full digitized liquidus as ground truth. At each iteration, each strategy picks the next composition to "measure," the model is refit with only the accumulated measurements, and the true MAE against the hidden full liquidus is recorded.

## Strategies

| Strategy | Selection rule |
|---|---|
| **FIM (D-optimal)** | Select the composition that maximally increases det(FIM) — most informative given current parameter estimates |
| **Random** | Sample uniformly at random from remaining digitized compositions (averaged over N trials) |
| **Binary search** | Van der Corput sequence (base 2) mapped to composition range: 0.5, 0.25, 0.75, 0.125, 0.625, ... |

All strategies start from the same DFT pseudo-constraint initial prediction and refit using only seen points.
The **asymptote** is the best MAE achieved by standard `fit_parameters` on the complete digitized liquidus.

In [1]:
import os
import copy
import random
import numpy as np
import plotly.graph_objects as go
from pathlib import Path

import gliquid.config as cfg
from gliquid.binary import BinaryLiquid, BLPlotter
from gliquid.fisher_information import compute_fim, find_optimal_next_measurement, build_nm_path_parameter_precision

os.environ["NEW_MP_API_KEY"] = "Jcw46im7UV1xOfHzbZZ8nkq8BH00Pf6s"

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

SYSTEM          = "Cu-Mg"    # Any system with a digitized liquidus
PARAM_FORMAT    = 'comb-exp'
N_ITER          = 10         # Measurements per strategy
N_RANDOM_TRIALS = 3          # Independent random-sampling runs (for mean ± std)
SIGMA           = 5.0        # K, assumed measurement uncertainty
PRIOR_LAMBDA    = 1e-4       # FIM cold-start regularization (first iteration)
NM_PATH_PRIOR_STRENGTH = 0.15  # Blend factor for trajectory-informed parameter precision
RANDOM_SEED     = 42
SAVE_FIGURES    = True       # Save per-iteration phase diagram HTML to disk

print(f"System: {SYSTEM}  |  {N_ITER} iterations  |  {N_RANDOM_TRIALS} random trials")

System: Cu-Mg  |  10 iterations  |  3 random trials


In [3]:
# ============================================================
# STEP 1: Load system, generate initial prediction, establish asymptote
# ============================================================

bl_base = BinaryLiquid.from_cache(SYSTEM, param_format=PARAM_FORMAT)
bl_base.comp_range_fit_lim = 0.0
bl_base.init_error = False

# Ground-truth reference: never modified, used only for MAE evaluation
reference_liq = list(bl_base.digitized_liq)
ref_max_liq_temp = max(pt[1] for pt in reference_liq)
ref_min_liq_temp = min(pt[1] for pt in reference_liq)
x_range = (min(pt[0] for pt in reference_liq if 0 < pt[0] < 1),
           max(pt[0] for pt in reference_liq if 0 < pt[0] < 1))
CANDIDATE_GRID = np.linspace(x_range[0], x_range[1], 100)

print(f"Loaded {SYSTEM}: {len(reference_liq)} reference liquidus points, "
      f"x ∈ [{x_range[0]:.3f}, {x_range[1]:.3f}]")

# --- Initial prediction: DFT pseudo-constraint (no invariant constraints) ---
# from gliquid.production_model_runner import ProductionModelRunner
# runner = ProductionModelRunner(str(cfg.data_dir) + "/20260217_135723")
# # predict_system returns [L0_a, L0_b, L1_a]; L1_b = 0 for comb-exp format
# pred_params = runner.predict_system(SYSTEM) + [0]
pred_params = [-27000, -5.7, -17000, 0]
bl_base.update_params(pred_params)
initial_params = bl_base.get_params()
BLPlotter(bl_base).show('pred+liq')
print(f"Initial params: L0_a={initial_params[0]:.1f}  L0_b={initial_params[1]:.4f}  "
      f"L1_a={initial_params[2]:.1f}  L1_b={initial_params[3]:.4f}")

# --- MAE of the initial prediction (iteration 0 baseline, shared by all strategies) ---
bl_init_eval = copy.deepcopy(bl_base)
bl_init_eval.digitized_liq   = reference_liq
bl_init_eval.max_liq_temp    = ref_max_liq_temp
bl_init_eval.min_liq_temp    = ref_min_liq_temp
bl_init_eval.ignored_comp_ranges = []
bl_init_eval.update_phase_points()
INITIAL_MAE, _, INITIAL_MAPE, _ = bl_init_eval.calculate_deviation_metrics()
print(f"Initial prediction MAE: {INITIAL_MAE:.2f} K  |  MAPE: {INITIAL_MAPE:.2f} %")

# --- Asymptote: best MAE from standard full-data fitting ---
print("\nComputing full-fit asymptote (standard fitting on complete liquidus)...")
bl_full = copy.deepcopy(bl_base)
bl_full.digitized_liq        = reference_liq
bl_full.max_liq_temp         = ref_max_liq_temp
bl_full.min_liq_temp         = ref_min_liq_temp
bl_full.ignored_comp_ranges  = []
full_fit_results = bl_full.fit_parameters(n_opts=5, max_iter=64, verbose=False)
ASYMPTOTE_MAE  = min(r['mae']  for r in full_fit_results)
ASYMPTOTE_MAPE = min(r['mape'] for r in full_fit_results)
print(f"Full-fit asymptote:  MAE = {ASYMPTOTE_MAE:.2f} K  |  MAPE = {ASYMPTOTE_MAPE:.2f} %")

Cu: H_liq = 13260 J/mol, S_liq = 9.7660 J/(mol·K), T_fusion = 1357.77 K, polymorphs = 0
Mg: H_liq = 8480 J/mol, S_liq = 9.1874 J/(mol·K), T_fusion = 923 K, polymorphs = 0

Reading MPDS json from entry at https://mpds.io/entry/C906729...

Loaded Cu-Mg: 103 reference liquidus points, x ∈ [0.000, 1.000]


Initial params: L0_a=-27000.0  L0_b=-5.7000  L1_a=-17000.0  L1_b=0.0000
Initial prediction MAE: 120.03 K  |  MAPE: 13.35 %

Computing full-fit asymptote (standard fitting on complete liquidus)...

--- Low temperature phase mismatch ---
MPDS: [||||||                           |                                |                                  ]
MP:   [|                                |                                |                                 |]
COMP:  0         10        20        30        40        50        60        70        80        90        100
--- Low temperature phases including component solid solutions ---
{'type': 'ss', 'name': '(Cu)', 'comp': 0.000381243, 'cbounds': [[0.000381243, 673.505], [0.0669068, 994.389]], 'tbounds': [[0.000381243, 673.505], [0.000381243, 1352.52]]}
{'type': 'lc', 'name': 'MgCu2', 'comp': 0.332825, 'tbounds': [[0.332825, 673.15], [0.332825, 1086.657]]}
{'type': 'lc', 'name': 'Mg2Cu', 'comp': 0.6666030000000001, 'tbounds': [[0.66660300000000

In [4]:
# ============================================================
# STEP 2: Helper functions
# ============================================================

def van_der_corput(n, x_min, x_max):
    """Van der Corput sequence (base 2) mapped to [x_min, x_max].
    Produces: 0.5, 0.25, 0.75, 0.125, 0.625, 0.375, 0.875, ...
    """
    points = []
    for i in range(1, n + 1):
        f, r, j = 1.0, 0.0, i
        while j > 0:
            f /= 2
            r += f * (j % 2)
            j //= 2
        points.append(x_min + r * (x_max - x_min))
    return points


def find_nearest_unseen(target_x, remaining):
    """Index in `remaining` of the point with composition nearest to target_x."""
    return int(np.argmin([abs(pt[0] - target_x) for pt in remaining]))


def refit_and_eval(seen):
    """Refit from bl_base using only `seen` points; evaluate MAE/MAPE vs. reference_liq.

    Two implementation details that affect FIM correctness:

    1. SORTING: `seen` must be sorted by composition before assigning to digitized_liq.
       fit_parameters reads digitized_liq[0][0] as the left bound and digitized_liq[-1][0]
       as the right bound. An unsorted list (e.g. appended in discovery order) inverts the
       range, causing every f() call to return inf and Nelder-Mead to fail silently.

    2. FALLBACK ON FITTING FAILURE: fit_parameters does not raise when NM fails — it
       returns [] and leaves _params in whatever intermediate state the last f() call
       set. If the return value is not checked, bl._params becomes garbage, and the
       FIM in the next iteration is computed at nonsensical parameters, causing
       recommendations that do not update with the measurement history.
       Fix: explicitly restore initial_params when fit_parameters returns [].

    Returns (fitted_bl, mae, mape, fit_succeeded: bool).
    """
    assert len(seen) > 0, "seen must contain at least one point"

    seen_sorted = sorted(seen, key=lambda pt: pt[0])  # ascending composition — required

    bl_fit = copy.deepcopy(bl_base)
    bl_fit.digitized_liq      = seen_sorted
    bl_fit.max_liq_temp       = ref_max_liq_temp
    bl_fit.min_liq_temp       = ref_min_liq_temp
    bl_fit.ignored_comp_ranges = []
    bl_fit.comp_range_fit_lim  = 0.0
    bl_fit.init_error          = False

    assert len(bl_fit.digitized_liq) == len(seen), (
        f"digitized_liq length {len(bl_fit.digitized_liq)} != seen length {len(seen)}")

    fit_results = bl_fit.fit_parameters(
        disable_inv_constrs=True,
        allow_sparse_data=True,
        check_liquidus_continuity=False,
        check_phase_mismatch=False,
        check_lupis_elliott=False,
        n_opts=1,
        max_iter=128,
        verbose=False,
        params_init=initial_params,
    )

    if not fit_results:
        # fit_parameters returns [] when NM cannot find physical parameter values.
        # _params is in an indeterminate intermediate state — restore a clean baseline
        # so the next FIM call is computed at known reasonable parameters.
        bl_fit.update_params(initial_params)   # resets _params and calls update_phase_points()
        fit_succeeded = False
    else:
        fit_succeeded = True

    # Evaluate against the FULL reference liquidus (never seen during fitting)
    bl_eval = copy.deepcopy(bl_fit)
    bl_eval.digitized_liq      = reference_liq
    bl_eval.max_liq_temp       = ref_max_liq_temp
    bl_eval.min_liq_temp       = ref_min_liq_temp
    bl_eval.ignored_comp_ranges = []
    bl_eval.update_phase_points()
    mae, _, mape, _ = bl_eval.calculate_deviation_metrics()
    return bl_fit, float(mae), float(mape), fit_succeeded


def save_iteration_plot(bl, strategy_dir, iteration, seen):
    """Save per-iteration phase diagram as HTML."""
    blp = BLPlotter(bl)
    fig = blp.get_plot('pred')


    fig.add_trace(go.Scatter(
        x=[pt[0] * 100 for pt in reference_liq],
        y=[pt[1] - 273.15 for pt in reference_liq],
        mode='markers',
        marker=dict(color='lightgray', size=5),
        name='Full reference (hidden)',
        opacity=0.5,
    ))

    if seen:
        fig.add_trace(go.Scatter(
            x=[pt[0] * 100 for pt in seen],
            y=[pt[1] - 273.15 for pt in seen],
            mode='markers',
            marker=dict(color='cornflowerblue', size=12, symbol='square',
                        line=dict(width=1, color='black')),
            name=f'Measured ({len(seen)} pts)',
        ))

    fig.update_layout(title=f"{SYSTEM} — {strategy_dir.name} — iteration {iteration + 1}")
    fig.write_html(str(strategy_dir / f"iter_{iteration:02d}.html"))


def run_strategy(strategy_name, sample_fn, output_dir, save_plots=True):
    """Generic sequential-measurement loop.

    sample_fn(iteration, seen, remaining, bl_current) -> int (index into remaining)

    Returns list of {'n': int, 'mae': float, 'mape': float} per iteration,
    with n=0 (initial prediction, no measurements) prepended.
    """
    remaining  = list(reference_liq)
    seen       = []
    history    = [{'n': 0, 'mae': INITIAL_MAE, 'mape': INITIAL_MAPE}]
    bl_current = copy.deepcopy(bl_base)   # starts from initial prediction params

    print(f"  Running {strategy_name}...")
    for i in range(N_ITER):
        idx = sample_fn(i, seen, remaining, bl_current)
        seen.append(remaining.pop(idx))

        bl_current, mae, mape, fit_ok = refit_and_eval(seen)
        history.append({'n': len(seen), 'mae': mae, 'mape': mape})

        if save_plots and SAVE_FIGURES:
            save_iteration_plot(bl_current, output_dir, i, seen)

        params = bl_current.get_params()
        status = "✓" if fit_ok else "✗ fallback"
        print(f"    iter {i+1:2d} | n={len(seen)} | x={seen[-1][0]:.3f} | "
              f"MAE={mae:.1f} K | MAPE={mape:.2f} %  [{status}]  "
              f"L0_b={params[1]:.3f}  L1_a={params[2]:.1f}")

    return history


print("Helper functions defined.")

Helper functions defined.


In [5]:
# ============================================================
# STEP 3: Create output directories
# ============================================================

OUTPUT_DIR = Path(cfg.project_root) / "figures" / f"benchmark_{SYSTEM.replace('-', '')}"
for name in ['fim', 'random', 'binary_search']:
    (OUTPUT_DIR / name).mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")

Output directory: c:\Users\willwerj\University of Michigan Dropbox\Joshua Willwerth\WHSun_Lab\G_liquid\gliquid_python\figures\benchmark_CuMg


In [ ]:
# ============================================================
# STEP 4a: FIM (D-optimal) strategy
# ============================================================

def fim_sample_fn(i, seen, remaining, bl):
    """Select the composition with the highest D-optimal gain given current FIM.

    The first measurement uses the cold-start prior only. Once a fit has been
    completed, the Nelder-Mead path is converted into a diagonal precision prior
    so the next FIM reflects both the measured liquidus and the observed fit
    trajectory. That keeps the acquisition tied to the current constraints while
    still letting the earlier parameter variation inform uncertainty.
    """
    x_seen = np.array([pt[0] for pt in seen]) if seen else np.array([])

    bl_fim = copy.deepcopy(bl)
    bl_fim.digitized_liq       = reference_liq
    bl_fim.max_liq_temp        = ref_max_liq_temp
    bl_fim.min_liq_temp        = ref_min_liq_temp
    bl_fim.ignored_comp_ranges = []
    bl_fim.update_phase_points()

    nm_prior = build_nm_path_parameter_precision(bl_fim)
    fim_kwargs = dict(
        x_compositions=CANDIDATE_GRID,
        sigma=SIGMA,
        prior_lambda=PRIOR_LAMBDA,
    )
    if nm_prior is not None:
        fim_kwargs['parameter_prior_precision'] = nm_prior
        fim_kwargs['parameter_prior_strength'] = NM_PATH_PRIOR_STRENGTH

    fim = compute_fim(bl_fim, **fim_kwargs)
    opt = find_optimal_next_measurement(fim, bl_fim, candidate_x=CANDIDATE_GRID)

    if not hasattr(fim_sample_fn, "diagnostics"):
        fim_sample_fn.diagnostics = []
    fim_sample_fn.diagnostics.append({
        'iteration': i + 1,
        'n_seen': len(seen),
        'n_out_of_range': opt.n_out_of_range,
        'top_x': float(opt.ranked_x[0]),
        'top_score': float(opt.d_optimal_scores[0]),
        'n_free_params': len(fim.param_names),
        'fim_det': float(fim.det_fim),
        'fim_condition_number': float(fim.condition_number),
        'fim_param_names': tuple(fim.param_names),
        'nm_prior_strength': float(NM_PATH_PRIOR_STRENGTH if nm_prior is not None else 0.0),
    })

    return find_nearest_unseen(opt.ranked_x[0], remaining)

print("=== FIM (D-optimal) ===")
fim_sample_fn.diagnostics = []
fim_history = run_strategy('fim', fim_sample_fn, OUTPUT_DIR / 'fim')

fim_diag_arr = np.array([row['n_out_of_range'] for row in fim_sample_fn.diagnostics], dtype=int)
print(f"FIM out-of-range candidate count per iteration: {fim_diag_arr.tolist()}")
print(f"FIM iterations with all candidates invalid: {int(np.sum(fim_diag_arr == len(CANDIDATE_GRID)))} / {len(fim_diag_arr)}")
print(f"FIM free parameters per iteration: {[row['n_free_params'] for row in fim_sample_fn.diagnostics]}")
print(f"FIM parameter names per iteration: {[row['fim_param_names'] for row in fim_sample_fn.diagnostics]}")
print(f"FIM condition numbers: {[round(row['fim_condition_number'], 2) for row in fim_sample_fn.diagnostics]}")
print(f"FIM NM-prior strengths: {[row['nm_prior_strength'] for row in fim_sample_fn.diagnostics]}")

=== FIM (D-optimal) ===
  Running fim...

Maximum composition range fitted: [0.9293399999999999, 0.9293399999999999]
Ignored composition ranges: []

Initial triangle for pseudo-constraints: [[-45725.66035313753, 7140.011039375004], [-45725.66035313753, -4554.419403409718], [-28260.012256272516, -4554.419403409718]]
    iter  1 | n=1 | x=0.929 | MAE=94.2 K | MAPE=10.08 %  [✓]  L0_b=-3.335  L1_a=-6768.0

Maximum composition range fitted: [0.865344, 0.9293399999999999]
Ignored composition ranges: []

Initial triangle for pseudo-constraints: [[-45725.66035313753, 7140.011039375004], [-45725.66035313753, -4554.419403409718], [-28260.012256272516, -4554.419403409718]]
    iter  2 | n=2 | x=0.865 | MAE=459.3 K | MAPE=47.85 %  [✓]  L0_b=-11.384  L1_a=-28512.3

Maximum composition range fitted: [0.332787, 0.9293399999999999]
Ignored composition ranges: []

Initial triangle for pseudo-constraints: [[-45725.66035313753, 7140.011039375004], [-45725.66035313753, -4554.419403409718], [-28260.0122562

In [ ]:
# ============================================================
# STEP 4b: Random strategy (N_RANDOM_TRIALS independent runs)
# ============================================================

rng = random.Random(RANDOM_SEED)
random_histories = []

for trial in range(N_RANDOM_TRIALS):
    # Capture trial-specific rng state via default-argument binding
    def random_sample_fn(i, seen, remaining, bl, _rng=rng):
        return _rng.randrange(len(remaining))

    print(f"=== Random — trial {trial + 1}/{N_RANDOM_TRIALS} ===")
    # Only save plots for the first trial to avoid disk overuse
    hist = run_strategy(
        f'random_trial{trial}',
        random_sample_fn,
        OUTPUT_DIR / 'random',
        save_plots=(trial == 0),
    )
    random_histories.append(hist)

random_mae_arr  = np.array([[h['mae']  for h in hist] for hist in random_histories])
random_mape_arr = np.array([[h['mape'] for h in hist] for hist in random_histories])
random_mae_mean = random_mae_arr.mean(axis=0)
random_mae_std  = random_mae_arr.std(axis=0)
print(f"\nRandom mean final MAE: {random_mae_mean[-1]:.2f} ± {random_mae_std[-1]:.2f} K")

=== Random — trial 1/3 ===
  Running random_trial0...

Maximum composition range fitted: [0.8175749999999999, 0.8175749999999999]
Ignored composition ranges: []

Initial triangle for pseudo-constraints: [[-45725.66035313753, 7140.011039375004], [-45725.66035313753, -4554.419403409718], [-28260.012256272516, -4554.419403409718]]
    iter  1 | n=1 | x=0.818 | MAE=155.5 K | MAPE=15.87 %  [✓]  L0_b=-5.675  L1_a=-27089.9

Maximum composition range fitted: [0.184588, 0.8175749999999999]
Ignored composition ranges: []

Initial triangle for pseudo-constraints: [[-45725.66035313753, 7140.011039375004], [-45725.66035313753, -4554.419403409718], [-28260.012256272516, -4554.419403409718]]
    iter  2 | n=2 | x=0.185 | MAE=52.0 K | MAPE=5.36 %  [✓]  L0_b=-4.221  L1_a=-3554.3

Maximum composition range fitted: [0.042360800000000004, 0.8175749999999999]
Ignored composition ranges: []

Initial triangle for pseudo-constraints: [[-45725.66035313753, 7140.011039375004], [-45725.66035313753, -4554.4194034

In [ ]:
# ============================================================
# STEP 4c: Binary search (Van der Corput) strategy
# ============================================================

bisect_xs = van_der_corput(N_ITER, *x_range)
print(f"Binary search sequence:")
print("  " + "  ".join(f"{x:.3f}" for x in bisect_xs))

def bisect_sample_fn(i, seen, remaining, bl):
    return find_nearest_unseen(bisect_xs[i], remaining)

print("\n=== Binary search ===")
bisect_history = run_strategy('binary_search', bisect_sample_fn, OUTPUT_DIR / 'binary_search')

Binary search sequence:
  0.500  0.250  0.750  0.125  0.625  0.375  0.875  0.063  0.562  0.313

=== Binary search ===
  Running binary_search...

Maximum composition range fitted: [0.504567, 0.504567]
Ignored composition ranges: []

Initial triangle for pseudo-constraints: [[-45725.66035313753, 7140.011039375004], [-45725.66035313753, -4554.419403409718], [-28260.012256272516, -4554.419403409718]]
    iter  1 | n=1 | x=0.505 | MAE=44.5 K | MAPE=5.10 %  [✓]  L0_b=-5.100  L1_a=-1216.0

Maximum composition range fitted: [0.247438, 0.504567]
Ignored composition ranges: []

Initial triangle for pseudo-constraints: [[-45725.66035313753, 7140.011039375004], [-45725.66035313753, -4554.419403409718], [-28260.012256272516, -4554.419403409718]]
    iter  2 | n=2 | x=0.247 | MAE=23.0 K | MAPE=2.30 %  [✓]  L0_b=-5.533  L1_a=-11423.8

Maximum composition range fitted: [0.247438, 0.751844]
Ignored composition ranges: []

Initial triangle for pseudo-constraints: [[-45725.66035313753, 7140.011039375004

In [ ]:
# ============================================================
# STEP 5: Summary figure
# ============================================================

n_axis    = [h['n']   for h in fim_history]          # 0, 1, 2, ..., N_ITER
fim_maes  = [h['mae'] for h in fim_history]
bis_maes  = [h['mae'] for h in bisect_history]

# Random: all histories share the same n=0 entry; stack only the per-trial MAEs
random_mae_arr  = np.array([[h['mae']  for h in hist] for hist in random_histories])
random_mae_mean = random_mae_arr.mean(axis=0)
random_mae_std  = random_mae_arr.std(axis=0)

fig = go.Figure()

# --- Shared initial prediction point (n=0, before any measurements) ---
fig.add_trace(go.Scatter(
    x=[0], y=[INITIAL_MAE],
    mode='markers',
    marker=dict(color='black', size=12, symbol='diamond'),
    name=f'Initial prediction ({INITIAL_MAE:.1f} K)',
    showlegend=True,
))

# --- FIM (D-optimal) ---
fig.add_trace(go.Scatter(
    x=n_axis, y=fim_maes,
    mode='lines+markers',
    name='FIM (D-optimal)',
    line=dict(color='#4477AA', width=2),
    marker=dict(size=8),
))

# --- Random ± 1σ band ---
fig.add_trace(go.Scatter(
    x=n_axis, y=random_mae_mean + random_mae_std,
    mode='lines', line=dict(width=0), showlegend=False,
))
fig.add_trace(go.Scatter(
    x=n_axis, y=random_mae_mean - random_mae_std,
    mode='lines', fill='tonexty',
    fillcolor='rgba(238,102,119,0.20)',
    line=dict(width=0), showlegend=False,
))
fig.add_trace(go.Scatter(
    x=n_axis, y=random_mae_mean,
    mode='lines+markers',
    name=f'Random (n={N_RANDOM_TRIALS} trials, mean ± 1σ)',
    line=dict(color='#EE6677', width=2),
    marker=dict(size=8),
))

# --- Binary search ---
fig.add_trace(go.Scatter(
    x=n_axis, y=bis_maes,
    mode='lines+markers',
    name='Binary search (Van der Corput)',
    line=dict(color='#228833', width=2),
    marker=dict(size=8),
))

# --- Full-fit asymptote ---
fig.add_hline(
    y=ASYMPTOTE_MAE,
    line_dash='dash', line_color='black', line_width=1.5,
    annotation_text=f'Full-fit asymptote ({ASYMPTOTE_MAE:.1f} K)',
    annotation_position='bottom right',
)

fig.update_layout(
    title=f'{SYSTEM}: Sequential measurement strategy comparison',
    xaxis=dict(title='Number of measurements', dtick=1, range=[-0.3, N_ITER]),
    yaxis_title='MAE to full liquidus (K)',
    template='simple_white',
    height=520,
    legend=dict(orientation='h', yanchor='top', y=-0.18, xanchor='left', x=0),
)
fig.show()

if SAVE_FIGURES:
    fig.write_html(str(OUTPUT_DIR / 'summary.html'))
    print(f"Summary saved to {OUTPUT_DIR / 'summary.html'}")

Summary saved to c:\Users\willwerj\University of Michigan Dropbox\Joshua Willwerth\WHSun_Lab\G_liquid\gliquid_python\figures\benchmark_CuMg\summary.html


In [ ]:
# ============================================================
# Results table
# ============================================================

from IPython.display import display, Markdown

rows = ['| Iter | x (FIM) | MAE FIM | x (bisect) | MAE bisect | MAE random (mean) |',
        '|------|---------|---------|------------|------------|-------------------|']

# Reconstruct sampled x-values for display (requires re-running or storing them)
# We only have MAE history here; show that.
for i in range(N_ITER):
    rows.append(
        f"| {i+1} "
        f"| — "
        f"| {fim_history[i]['mae']:.1f} K "
        f"| — "
        f"| {bisect_history[i]['mae']:.1f} K "
        f"| {random_mae_mean[i]:.1f} ± {random_mae_std[i]:.1f} K |"
    )
rows.append(f"| **Asymptote** | — | **{ASYMPTOTE_MAE:.1f} K** | — | **{ASYMPTOTE_MAE:.1f} K** | **{ASYMPTOTE_MAE:.1f} K** |")

display(Markdown('\n'.join(rows)))

| Iter | x (FIM) | MAE FIM | x (bisect) | MAE bisect | MAE random (mean) |
|------|---------|---------|------------|------------|-------------------|
| 1 | — | 120.0 K | — | 120.0 K | 120.0 ± 0.0 K |
| 2 | — | 94.2 K | — | 44.5 K | 77.8 ± 55.4 K |
| 3 | — | 459.3 K | — | 23.0 K | 169.7 ± 192.6 K |
| 4 | — | 38.2 K | — | 20.0 K | 35.8 ± 14.2 K |
| 5 | — | 29.5 K | — | 20.0 K | 37.5 ± 14.7 K |
| 6 | — | 28.8 K | — | 20.0 K | 50.5 ± 27.6 K |
| 7 | — | 39.6 K | — | 20.0 K | 25.8 ± 10.9 K |
| 8 | — | 51.7 K | — | 20.0 K | 18.2 ± 1.6 K |
| 9 | — | 42.2 K | — | 20.0 K | 18.1 ± 1.7 K |
| 10 | — | 41.9 K | — | 20.0 K | 27.8 ± 15.1 K |
| **Asymptote** | — | **15.8 K** | — | **15.8 K** | **15.8 K** |